# Momentum Gap Scanner — starter notebook

This notebook loads the Parquet lake with DuckDB and works one question end to
end, applying the three corrections that decide whether a number from this lake
means anything:

1. **Re-weight the control sample.** Tier 1 keeps every mover and only a
   `control_sample_pct` slice of everything else. A rate computed without
   weighting the control rows back up has the wrong denominator and will look
   far better than reality.
2. **Report tradeable and all rows side by side.** A +40% move in a 2M-float
   name with a 30-cent spread is not +40% in an account.
3. **Exclude `suspect_price` and `spans_halt` rows.** Those are not conservative
   estimates, they are meaningless numbers.

Work the questions in the order of the research roadmap: when moves actually
start, then each pillar's base rate on its own, then how the pillars correlate,
and only then joint rules.


In [ ]:
import duckdb
from pathlib import Path

LAKE = Path("../data/lake")
CONTROL_SAMPLE_PCT = 10          # must match config.yaml
CONTROL_WEIGHT = 100 / CONTROL_SAMPLE_PCT

con = duckdb.connect()
for table in ["daily_universe", "snapshots", "reference", "evaluations",
              "news", "outcomes", "runners", "pruned_summary", "bars_1m"]:
    path = LAKE / table
    if path.exists():
        con.execute(
            f"CREATE OR REPLACE VIEW {table} AS "
            f"SELECT * FROM read_parquet('{path}/**/*.parquet', hive_partitioning = true)"
        )

con.execute("SELECT table_name FROM information_schema.tables ORDER BY 1").df()

## 1. How much history is there?

Sample size governs everything below. Roughly 40 movers a day is ~10,000
mover-days a year, but conditioned on float < 20M **and** \$2–20 **and** fresh
news it falls to about 1–3 a day. Estimating a hit rate to ±5% needs a few
hundred observations: weeks for loose single-pillar questions, about a year
before trusting anything about the full five-pillar setup.

In [ ]:
con.execute("""
    SELECT 'evaluations' AS table_name, count(DISTINCT date) AS days, count(*) AS rows FROM evaluations
    UNION ALL SELECT 'snapshots', count(DISTINCT date), count(*) FROM snapshots
    UNION ALL SELECT 'runners', count(DISTINCT date), count(*) FROM runners
    UNION ALL SELECT 'outcomes', count(DISTINCT date), count(*) FROM outcomes
    ORDER BY 1
""").df()

## 2. The control re-weighting, worked

The question: **of the stocks that met a candidate setup at 08:05, what
fraction actually ran?**

The numerator is easy. The denominator is the trap: `snapshots` keeps every
mover but only ~10% of the names that went nowhere, so counting rows directly
inflates the rate by roughly 10x for the uninteresting population.

Each retained row carries `retention_class`. Movers and signals count once;
control rows count `CONTROL_WEIGHT` times.

In [ ]:
setup = con.execute(f"""
    WITH per_ticker AS (
        SELECT
            s.date,
            s.ticker,
            any_value(s.retention_class)                     AS retention_class,
            max(s.rvol)                                      AS max_rvol,
            max(s.gap_pct)                                   AS max_gap_pct,
            max(r.float_shares_outstanding)                  AS float_shares
        FROM snapshots s
        LEFT JOIN reference r ON r.ticker = s.ticker AND r.date = s.date
        WHERE NOT coalesce(s.suspect_price, false)
        GROUP BY 1, 2
    ),
    labelled AS (
        SELECT
            p.*,
            CASE WHEN p.retention_class = 'control' THEN {CONTROL_WEIGHT} ELSE 1 END AS weight,
            EXISTS (SELECT 1 FROM runners x WHERE x.ticker = p.ticker AND x.date = p.date) AS ran
        FROM per_ticker p
    )
    SELECT
        count(*)                                   AS rows_kept,
        sum(weight)                                AS estimated_population,
        sum(CASE WHEN ran THEN weight ELSE 0 END)  AS estimated_runners,
        sum(CASE WHEN ran THEN weight ELSE 0 END) / nullif(sum(weight), 0) AS hit_rate
    FROM labelled
    WHERE max_rvol >= 5 AND float_shares < 20000000
""").df()
setup

Read it this way: `rows_kept` is what is on disk, `estimated_population` is
what that represents. The unweighted rate — `estimated_runners / rows_kept` —
is the number to distrust; it is the single easiest mistake to make with this
lake.

In [ ]:
if not setup.empty and setup.loc[0, "rows_kept"]:
    naive = setup.loc[0, "estimated_runners"] / setup.loc[0, "rows_kept"]
    print(f"weighted hit rate : {setup.loc[0, 'hit_rate']:.1%}")
    print(f"naive (wrong)     : {naive:.1%}")

## 3. Tradeable versus all rows

Every outcome row carries `tradeable`, `dollar_volume_in_window` and
`est_spread_pct`. Report both figures, always. A rule that looks profitable
only on rows nobody could have filled is not a rule.

In [ ]:
con.execute("""
    SELECT
        tradeable,
        count(*)            AS n,
        avg(ret_30m_pct)    AS avg_ret_30m,
        avg(mfe_pct)        AS avg_mfe,
        avg(mae_pct)        AS avg_mae
    FROM outcomes
    WHERE NOT coalesce(spans_halt, false)
    GROUP BY 1
    ORDER BY 1
""").df()

`spans_halt` rows are excluded rather than included with a caveat: a return
measured across a halt is not a noisy estimate of anything, and a stock can
reopen 40% higher in a single print.

## 4. When do moves actually start?

The highest value per unit of effort in the whole project. The 08:00 / 08:30 /
09:00 windows are a guess; within three or four weeks this shows whether moves
actually cluster there. Expect 09:00–09:30 and the 16:00–16:15 earnings slot to
dominate, and expect at least one morning window to be dead weight.

In [ ]:
con.execute("""
    SELECT
        strftime(move_start_utc AT TIME ZONE 'America/New_York', '%H:%M') AS move_start_et,
        count(*) AS runners
    FROM runners
    WHERE move_start_utc IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()

## 5. Base rate per pillar, individually

Before combining pillars, know which of them carry information at all. Each row
below is one pillar on its own, with the control re-weighting applied.

In [ ]:
con.execute(f"""
    WITH per_ticker AS (
        SELECT
            e.date, e.ticker,
            max(e.gap_pct) AS gap_pct, max(e.rvol) AS rvol,
            max(e.float_shares) AS float_shares,
            bool_or(e.pillar_3_status = 'pass') AS had_news,
            EXISTS (SELECT 1 FROM runners r WHERE r.ticker = e.ticker AND r.date = e.date) AS ran
        FROM evaluations e
        GROUP BY 1, 2
    )
    SELECT 'gap >= 10%'   AS pillar, avg(CASE WHEN ran THEN 1.0 ELSE 0 END) AS hit_rate, count(*) AS n
      FROM per_ticker WHERE gap_pct >= 10
    UNION ALL
    SELECT 'rvol >= 5',  avg(CASE WHEN ran THEN 1.0 ELSE 0 END), count(*)
      FROM per_ticker WHERE rvol >= 5
    UNION ALL
    SELECT 'float < 20M', avg(CASE WHEN ran THEN 1.0 ELSE 0 END), count(*)
      FROM per_ticker WHERE float_shares < 20000000
    UNION ALL
    SELECT 'fresh news',  avg(CASE WHEN ran THEN 1.0 ELSE 0 END), count(*)
      FROM per_ticker WHERE had_news
""").df()

These are unweighted on purpose — they run over `evaluations`, which is kept
in full, not over the pruned `snapshots`. Mixing the two sources in one rate is
another way to get the denominator wrong.

## 6. Setups I skipped that ran, and setups I took that faded

The journal is the only data here that cannot be reconstructed: it is the
trader's own context at the moment of the decision. Both queries are one join.

In [ ]:
con.execute("""
    SELECT j.date, j.ticker, j.action, j.note, r.high_of_day_pct
    FROM read_parquet('../data/lake/journal/**/*.parquet', hive_partitioning = true) j
    JOIN runners r ON r.ticker = j.ticker AND r.date = j.date
    WHERE j.action = 'skipped'
    ORDER BY r.high_of_day_pct DESC
""").df() if (LAKE / 'journal').exists() else 'no journal entries yet'

## 7. Before you trust any of this

- **Multiple testing.** Fifty threshold combinations against five hundred
  observations will produce two or three that look excellent by chance. Settle
  a rule on the earlier data, then test it on the holdout **once**. `/research`
  shows the cutoff and refuses to query past it unless you override.
- **Pillar correlation.** Gap and RVOL move together; price and float move
  together. If the effective dimensionality is 2–3 rather than 5, "5 of 5" is a
  weaker filter than it looks — and that is why so few names ever pass it.
- **What is missing is data too.** `pruned_summary` holds one row per dropped
  ticker, and `data_quality` records the days when a feed was down. A rate
  computed over a week with a broken news feed is a rate about the feed.

This tool screens stocks, records data and sends alerts. It does not provide
financial advice.